# Train the "Hey Nexus" wake word

Trains a robust [openWakeWord](https://github.com/dscripka/openWakeWord) model for the assistant: 50,000 synthetic voices saying your phrase, plus **your own recordings** of it, mixed with noise, room echo and your room's sound, and sound-alikes it must ignore. The result is one small file, `hey_nexus.onnx`, that goes in the assistant's `wakeword/` folder.

**Before you start**, on your PC: `python wakeword_tools.py myvoice` (about 10 minutes, makes `wakeword/my_voice.zip`) and, if you haven't yet, `python wakeword_tools.py noise 20` (makes `wakeword/my_room_noise.zip`).

**Run the cells top to bottom.** Every cell can be re-run safely: downloads resume, and finished steps are skipped.

**Where it runs:**
- **Google Colab:** choose **Runtime → Change runtime type → T4 GPU** (or L4). Not a TPU: training needs an NVIDIA GPU. Turn on `save_to_drive` in cell 1, so a disconnect or the session time limit doesn't cost you the hours of clip generation.
- **Your own PC (Windows):** run it in WSL2 Ubuntu with Jupyter. See **Running locally** at the bottom of this notebook. There's no time limit, and the files stay on disk between runs.

**Retraining** (for example, a wake word that's too strict or too jumpy): change `false_activation_penalty` in cell 1. In the same session, re-run cells 1, 6 and 8. In a new Colab session, run all the cells: cell 8 restores the clip features from your Drive checkpoint, so only the training step runs (under an hour instead of hours). The previous model is kept with a date in its name.

Based on openWakeWord's `automatic_model_training.ipynb` (Apache-2.0) and the current-Colab fixes from [openWakeWord PR #359](https://github.com/dscripka/openWakeWord/pull/359).

In [ ]:
# @title 1. Settings { display-mode: "form" }
# @markdown Spell the phrase the way you want it *pronounced*. Listen to it in cell 7 before training.
wake_phrase = "hey nexus"  # @param {type:"string"}
number_of_examples = 50000  # @param {type:"integer"}
training_steps = 50000  # @param {type:"integer"}
# @markdown How hard training punishes false wake-ups. Higher = stricter (misses more). 2000 came out too strict; 1000 is openWakeWord's default.
false_activation_penalty = 1000  # @param {type:"integer"}
augmentation_rounds = 2  # @param {type:"integer"}
# @markdown Local PC: paths to `my_room_noise.zip` and `my_voice.zip` (for example `/mnt/f/E-book/AIAssistant/wakeword/my_voice.zip`). Colab: leave empty and upload them in cell 5.
room_noise_zip = ""  # @param {type:"string"}
my_voice_zip = ""  # @param {type:"string"}
# @markdown Each of your recordings is used this many times (each time with different noise and echo). 40 copies of ~60 clips ≈ 5% of the training examples.
my_voice_copies = 40  # @param {type:"integer"}
add_music = False  # @param {type:"boolean"}
# @markdown Colab only: keep a checkpoint in Google Drive so a disconnect doesn't lose the long steps.
save_to_drive = True  # @param {type:"boolean"}

# Sound-alikes that must NOT wake the assistant (each repeated so it gets real weight).
# Don't add "nexis" or "next us": they sound the same as the wake phrase and would confuse the model.
SOUND_ALIKES = ["hey lexus", "lexus", "hey texas", "texas", "hey alexa", "alexa", "necklace",
                "hey next", "next", "hey nick", "hey rex", "hey max", "focus", "census",
                "hey becca", "hello"]

import os, sys
IN_COLAB = "google.colab" in sys.modules
BASE = "/content" if IN_COLAB else os.path.expanduser("~/wakeword_training")
os.makedirs(BASE, exist_ok=True)
MODEL = wake_phrase.strip().lower().replace(" ", "_")
OUT_DIR = f"{BASE}/model_{MODEL}"          # the trained .onnx lands here
FEAT_DIR = f"{OUT_DIR}/{MODEL}"            # clips and features
PY = f"{BASE}/py311/bin/python"            # the training environment's Python
CFG = f"{BASE}/{MODEL}.yaml"
OWW = f"{BASE}/openWakeWord"
PIPER = f"{BASE}/piper-sample-generator"
DRIVE_DIR = f"/content/drive/MyDrive/wakeword_training/{MODEL}"

if IN_COLAB and save_to_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Running on: {'Google Colab' if IN_COLAB else 'this PC'}  |  working folder: {BASE}")
print(f"Phrase: '{wake_phrase}'  →  model file: {MODEL}.onnx")

In [ ]:
# @title 2. Set up the training environment (10–15 min)
import glob, shutil, subprocess

# A GPU is required (a TPU or CPU runtime can't run the speech generator or training)
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, check=True).stdout.strip()
    print("GPU:", gpu)
except Exception:
    raise RuntimeError("No NVIDIA GPU found. In Colab: Runtime → Change runtime type → T4 GPU. "
                       "Locally: install the NVIDIA driver on Windows and check `nvidia-smi` works in WSL.")

# System packages
if os.geteuid() == 0:
    !apt-get update -qq
    !apt-get install -y -qq espeak-ng libespeak-ng-dev ffmpeg git curl
else:
    missing = [t for t in ("espeak-ng", "ffmpeg", "git", "curl") if not shutil.which(t)]
    if missing:
        raise RuntimeError(f"Missing {missing}. In the Ubuntu terminal run:\n"
                           "  sudo apt update && sudo apt install -y espeak-ng libespeak-ng-dev ffmpeg git curl")

# Python 3.11 environment via uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
UV = next((c for c in (shutil.which("uv"), os.path.expanduser("~/.local/bin/uv"),
                       os.path.expanduser("~/.cargo/bin/uv")) if c and os.path.exists(c)), "uv")
if not os.path.exists(PY):
    !{UV} python install 3.11
    !{UV} venv {BASE}/py311 --python 3.11
if not os.path.exists(PY):
    raise RuntimeError(f"Couldn't create the Python 3.11 environment at {BASE}/py311.")
PIP = f"{UV} pip install --python {BASE}/py311"

!{PIP} torch torchaudio --index-url https://download.pytorch.org/whl/cu121
DEPS = ('"scipy==1.13.1" "setuptools<70" "datasets<3" "pyarrow<17" '
        'mutagen soundfile audiomentations torch-audiomentations acoustics torchinfo torchmetrics '
        'speechbrain deep-phonemizer piper-phonemize espeak-phonemizer webrtcvad pronouncing '
        'onnx onnxruntime-gpu onnx2tf tensorflow tensorflow-probability torchcodec onnxscript psutil '
        'matplotlib matplotlib-inline')
!{PIP} {DEPS}
# The line above can swap in a torch or torchaudio that don't match each other
# ("libtorchaudio.so: undefined symbol"). Pin a matching pair last.
!{PIP} "torch==2.5.1" "torchaudio==2.5.1" --index-url https://download.pytorch.org/whl/cu121

if not os.path.exists(OWW):
    !git clone https://github.com/dscripka/openWakeWord.git {OWW}
if not os.path.exists(PIPER):
    !git clone https://github.com/dscripka/piper-sample-generator.git {PIPER}
!{PIP} --no-deps -e {OWW}

# Compatibility patches (from PR #359): PyTorch 2.6+ loading, torchaudio.info, headless matplotlib
sitecustomize_code = """
import os
os.environ["MPLBACKEND"] = "Agg"

import torch
import torch.serialization
import torchaudio
import soundfile as sf
from collections import namedtuple

_orig_load = torch.load
def _custom_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)
torch.load = _custom_load
torch.serialization.load = _custom_load

AudioMetaData = namedtuple("AudioMetaData", ["sample_rate", "num_frames", "num_channels", "bits_per_sample", "encoding"])
def _custom_info(filepath, *args, **kwargs):
    info = sf.info(filepath)
    return AudioMetaData(info.samplerate, info.frames, info.channels, 16, "PCM_S")
torchaudio.info = _custom_info
"""
site_pkgs = glob.glob(f"{BASE}/py311/lib/python3.11/site-packages")[0]
open(os.path.join(site_pkgs, "sitecustomize.py"), "w").write(sitecustomize_code)

init_path = f"{OWW}/openwakeword/__init__.py"
init_code = open(init_path).read()
if "_custom_load" not in init_code:
    open(init_path, "w").write(sitecustomize_code + "\n" + init_code)

data_path = f"{OWW}/openwakeword/data.py"
data_content = open(data_path).read()
if "rir_waveform, sr = torchaudio.load" in data_content:      # sample-rate name collision
    data_content = data_content.replace("rir_waveform, sr = torchaudio.load", "rir_waveform, _ = torchaudio.load")
    open(data_path, "w").write(data_content)
train_path = f"{OWW}/openwakeword/train.py"
train_content = open(train_path).read()
train_content = train_content.replace('default="False"', 'default=False')
open(train_path, "w").write(train_content)

# Check: torch + torchaudio load together and see the GPU
check = subprocess.run([PY, "-c", "import torch, torchaudio; print(torch.__version__, torchaudio.__version__, torch.cuda.is_available())"],
                       capture_output=True, text=True)
print(check.stdout.strip() or check.stderr[-1500:])
if check.returncode != 0 or not check.stdout.strip().endswith("True"):
    raise RuntimeError("torch/torchaudio didn't load with the GPU (see above). Re-run this cell; if it fails again, paste the error.")
print("Environment ready.")

In [ ]:
# @title 3. Download models (2–5 min)
import urllib.request

def npy_ok(path):
    """True if a .npy file is complete (catches downloads that stopped early)."""
    import numpy as np
    try:
        with open(path, "rb") as fh:
            version = np.lib.format.read_magic(fh)
            read = np.lib.format.read_array_header_1_0 if version == (1, 0) else np.lib.format.read_array_header_2_0
            shape, _, dtype = read(fh)
            need = fh.tell() + int(np.prod(shape)) * dtype.itemsize
        return os.path.getsize(path) >= need
    except Exception:
        return False

def fetch(url, dest, min_bytes):
    if not os.path.exists(dest) or os.path.getsize(dest) < min_bytes:
        print("Downloading", os.path.basename(dest))
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f)
    if os.path.getsize(dest) < min_bytes:
        raise RuntimeError(f"{dest} looks incomplete — re-run this cell.")

# openWakeWord's feature extractor, embedding model and voice-activity model
open(f"{BASE}/dl_models.py", "w").write("from openwakeword.utils import download_models\ndownload_models()\n")
!{PY} {BASE}/dl_models.py
for name in ("melspectrogram.onnx", "embedding_model.onnx"):
    if not os.path.exists(f"{OWW}/openwakeword/resources/models/{name}"):
        raise RuntimeError(f"{name} didn't download — see the error above.")

os.makedirs(f"{OWW}/openwakeword/resources", exist_ok=True)
fetch("https://huggingface.co/TaterTotterson/DeepPhonemizer/resolve/main/en_us_cmudict_forward.pt",
      f"{OWW}/openwakeword/resources/en_us_cmudict_forward.pt", 1_000)
os.makedirs(f"{PIPER}/models", exist_ok=True)
fetch("https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt",
      f"{PIPER}/models/en-us-libritts-high.pt", 1_000_000)
fetch("https://raw.githubusercontent.com/dscripka/piper-sample-generator/master/models/en-us-libritts-high.pt.json",
      f"{PIPER}/models/en-us-libritts-high.pt.json", 100)
print("Models ready.")

In [ ]:
# @title 4. Download training data (15–30 min, about 16 GB)
FEATURES_URL = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/"
for name in ("validation_set_features.npy", "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    path = f"{BASE}/{name}"
    if npy_ok(path):
        print(f"{name}: complete")
        continue
    # -C - resumes a partial download; --fail stops on HTTP errors instead of saving an error page
    !curl -L --fail -C - -# -o {path} "{FEATURES_URL}{name}"
    if not npy_ok(path):
        raise RuntimeError(f"{name} is incomplete — re-run this cell to resume the download.")
    print(f"{name}: complete")

# Room echo (MIT impulse responses) and everyday noise (ESC-50)
dl_audio = f"""
import os, soundfile as sf
from datasets import load_dataset
from tqdm import tqdm
os.makedirs("{BASE}/mit_rirs", exist_ok=True)
if len(os.listdir("{BASE}/mit_rirs")) < 250:
    ds = load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for i, row in enumerate(tqdm(ds, total=271, desc="room echoes")):
        sf.write(f"{BASE}/mit_rirs/{{i}}.wav", row["audio"]["array"], row["audio"]["sampling_rate"])
os.makedirs("{BASE}/background_clips", exist_ok=True)
if len(os.listdir("{BASE}/background_clips")) < 1900:
    ds = load_dataset("ashraq/esc50", split="train", streaming=True)
    for i, row in enumerate(tqdm(ds, total=2000, desc="noise clips")):
        sf.write(f"{BASE}/background_clips/{{i}}.wav", row["audio"]["array"], row["audio"]["sampling_rate"])
"""
open(f"{BASE}/dl_audio.py", "w").write(dl_audio)
!{PY} {BASE}/dl_audio.py
n_rir, n_bg = len(os.listdir(f"{BASE}/mit_rirs")), len(os.listdir(f"{BASE}/background_clips"))
print(f"{n_rir} room echoes, {n_bg} noise clips")
if n_rir < 250 or n_bg < 1900:
    raise RuntimeError("Echo/noise download incomplete — re-run this cell.")

In [ ]:
# @title 5. Your voice, your room noise (and optional music)
import zipfile
voice_dir = f"{BASE}/my_voice"
os.makedirs(voice_dir, exist_ok=True)
if not glob.glob(f"{voice_dir}/*.wav"):
    if my_voice_zip:
        zipfile.ZipFile(my_voice_zip).extractall(voice_dir)
    elif IN_COLAB:
        from google.colab import files
        print("Pick wakeword/my_voice.zip (made by: python wakeword_tools.py myvoice), or Cancel to skip")
        for name in files.upload():
            zipfile.ZipFile(name).extractall(voice_dir)
n_voice = len(glob.glob(f"{voice_dir}/*.wav"))
print(n_voice, "recordings of your voice", "(none — training uses synthetic voices only)" if not n_voice else
      f"→ {n_voice * my_voice_copies} training examples")

room_dir = f"{BASE}/my_room_noise"
os.makedirs(room_dir, exist_ok=True)
if not os.listdir(room_dir):
    if room_noise_zip:
        zipfile.ZipFile(room_noise_zip).extractall(room_dir)
    elif IN_COLAB:
        from google.colab import files
        print("Pick wakeword/my_room_noise.zip (made by: python wakeword_tools.py noise 20)")
        for name in files.upload():
            zipfile.ZipFile(name).extractall(room_dir)
print(len(os.listdir(room_dir)), "room clips", "(none — training still works, just less tuned to your room)" if not os.listdir(room_dir) else "")

if add_music and len(os.listdir(f"{BASE}/fma") if os.path.isdir(f"{BASE}/fma") else []) < 200:
    dl_music = f"""
import os, soundfile as sf
from datasets import load_dataset, Audio
os.makedirs("{BASE}/fma", exist_ok=True)
ds = load_dataset("rudraml/fma", name="small", split="train", streaming=True, trust_remote_code=True)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
for i, row in enumerate(ds):
    if i >= 240: break                          # 240 x 30 s = 2 hours of music
    sf.write(f"{BASE}/fma/{{i}}.wav", row["audio"]["array"], 16000, subtype="PCM_16")
"""
    open(f"{BASE}/dl_music.py", "w").write(dl_music)
    !{PY} {BASE}/dl_music.py
    print(len(os.listdir(f"{BASE}/fma")) if os.path.isdir(f"{BASE}/fma") else 0,
          "music clips (if this failed, set add_music off — it's optional)")

In [ ]:
# @title 6. Write the training configuration
import yaml
gpu_gb = float(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.split()[0]) / 1024
tts_batch, aug_batch = (150, 64) if gpu_gb >= 30 else (100, 32) if gpu_gb >= 15 else (50, 16)

background, rates = [f"{BASE}/background_clips"], [1]
for folder, rate in ((f"{BASE}/fma", 1), (f"{BASE}/my_room_noise", 3)):   # your room counts 3x
    if os.path.isdir(folder) and os.listdir(folder):
        background.append(folder); rates.append(rate)

config = {
    "model_name": MODEL,
    "target_phrase": [wake_phrase],
    "custom_negative_phrases": SOUND_ALIKES * 100,
    "n_samples": number_of_examples,
    "n_samples_val": max(500, number_of_examples // 10),
    "tts_batch_size": tts_batch,
    "augmentation_batch_size": aug_batch,
    "piper_sample_generator_path": PIPER,
    "output_dir": OUT_DIR,
    "rir_paths": [f"{BASE}/mit_rirs"],
    "background_paths": background,
    "background_paths_duplication_rate": rates,
    "false_positive_validation_data_path": f"{BASE}/validation_set_features.npy",
    "augmentation_rounds": augmentation_rounds,
    "feature_data_files": {"ACAV100M_sample": f"{BASE}/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
    "model_type": "dnn",
    "layer_size": 32,
    "steps": training_steps,
    "max_negative_weight": false_activation_penalty,
    "target_false_positives_per_hour": 0.2,
    "target_accuracy": 0.7,
    "target_recall": 0.5,
}
os.makedirs(OUT_DIR, exist_ok=True)
yaml.safe_dump(config, open(CFG, "w"))
print(f"Saved {CFG}")
print(f"  {number_of_examples} examples, {training_steps} steps, penalty {false_activation_penalty}, "
      f"{augmentation_rounds} augmentation rounds, GPU {gpu_gb:.0f} GB → batch {tts_batch}/{aug_batch}")
print("  background:", ", ".join(f"{os.path.basename(b)} x{r}" for b, r in zip(background, rates)))

In [ ]:
# @title 7. Listen to the pronunciation
from IPython.display import Audio, display
test_dir = f"{BASE}/test_pronunciation"
shutil.rmtree(test_dir, ignore_errors=True)
os.makedirs(test_dir)
!{PY} {PIPER}/generate_samples.py "{wake_phrase}" --max-samples 3 --model {PIPER}/models/en-us-libritts-high.pt --output-dir {test_dir}
for wav in sorted(glob.glob(f"{test_dir}/*.wav")):
    display(Audio(wav))
print("If they don't sound like you saying it, change wake_phrase in cell 1 (spell it phonetically) and re-run 1, 6 and 7.")

In [ ]:
# @title 8. Train (first run: 3–5 hours on a T4; retrains: training step only)
import time, zipfile
TRAIN = f"cd {BASE} && {PY} {OWW}/openwakeword/train.py --training_config {CFG}"
FEATURES = ["positive_features_train.npy", "negative_features_train.npy",
            "positive_features_test.npy", "negative_features_test.npy"]

def sh(cmd):
    """Run a command and show its output live (Colab hides output from plain subprocesses)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "MPLBACKEND": "Agg"})
    while chunk := p.stdout.read1(4096):
        sys.stdout.write(chunk.decode(errors="replace")); sys.stdout.flush()
    return p.wait()

def step(name, *flags):
    print(f"\n=== {name} ===", flush=True)
    if sh(f"{TRAIN} {' '.join(flags)}") != 0:
        raise RuntimeError(f"{name} failed — the error is just above.")

def features_ready():
    return all(npy_ok(f"{FEAT_DIR}/{f}") for f in FEATURES)

def n_wavs(sub):
    return len(glob.glob(f"{FEAT_DIR}/{sub}/*.wav"))

def save_checkpoint():
    if not (IN_COLAB and save_to_drive):
        return
    print("Saving a checkpoint to Google Drive...", flush=True)
    for f in FEATURES:
        shutil.copy(f"{FEAT_DIR}/{f}", DRIVE_DIR)
    if os.path.exists(VOICE_MARK):
        shutil.copy(VOICE_MARK, DRIVE_DIR)
    with zipfile.ZipFile(f"{DRIVE_DIR}/positive_test.zip", "w") as z:    # training samples 50 to measure clip length
        for wav in sorted(glob.glob(f"{FEAT_DIR}/positive_test/*.wav"))[:200]:
            z.write(wav, os.path.basename(wav))
    print("Checkpoint saved:", DRIVE_DIR)

VOICE_MARK = f"{FEAT_DIR}/my_voice_included.txt"     # how many of your clips the features include

def voice_wanted():
    return len(glob.glob(f"{BASE}/my_voice/*.wav")) * my_voice_copies

def voice_included():
    return int(open(VOICE_MARK).read()) if os.path.exists(VOICE_MARK) else 0

def add_my_voice():
    """Copy your recordings into the training clips (after the synthetic ones
    are generated, so the generator doesn't count them)."""
    for old in glob.glob(f"{FEAT_DIR}/positive_train/my_voice_*.wav"):
        os.remove(old)
    clips = sorted(glob.glob(f"{BASE}/my_voice/*.wav"))
    for i, clip in enumerate(clips):
        for k in range(my_voice_copies):
            shutil.copy(clip, f"{FEAT_DIR}/positive_train/my_voice_{i:04d}_{k:02d}.wav")
    print(f"Added {len(clips)} recordings of your voice x {my_voice_copies} = {len(clips) * my_voice_copies} examples")
    return len(clips) * my_voice_copies

def restore_checkpoint():
    if not (IN_COLAB and save_to_drive) or features_ready():
        return
    if all(npy_ok(f"{DRIVE_DIR}/{f}") for f in FEATURES) and os.path.exists(f"{DRIVE_DIR}/positive_test.zip"):
        print("Restoring clips' features from the Google Drive checkpoint...", flush=True)
        os.makedirs(f"{FEAT_DIR}/positive_test", exist_ok=True)
        for f in FEATURES:
            shutil.copy(f"{DRIVE_DIR}/{f}", FEAT_DIR)
        zipfile.ZipFile(f"{DRIVE_DIR}/positive_test.zip").extractall(f"{FEAT_DIR}/positive_test")
        if os.path.exists(f"{DRIVE_DIR}/my_voice_included.txt"):
            shutil.copy(f"{DRIVE_DIR}/my_voice_included.txt", VOICE_MARK)

restore_checkpoint()
if features_ready() and voice_included() != voice_wanted() and not glob.glob(f"{FEAT_DIR}/positive_train/*.wav"):
    print("Note: these saved features don't match your current voice recordings, and the clips needed to "
          "redo them aren't here (restored from a checkpoint). Training continues with the saved features.")
elif features_ready() and voice_included() != voice_wanted():
    print("Your voice recordings changed since the features were computed — recomputing them.")
    open(VOICE_MARK, "w").write(str(add_my_voice()))
    step("2/3 Adding noise and echo, computing features", "--augment_clips", "--overwrite")
    save_checkpoint()
if features_ready():
    print("Clip features are ready — skipping to training.")
else:
    step("1/3 Generating synthetic clips (resumes if interrupted)", "--generate_clips")
    n_val = max(500, number_of_examples // 10)
    counts = {d: n_wavs(d) for d in ("positive_train", "negative_train", "positive_test", "negative_test")}
    print("Clips:", counts)
    if min(counts["positive_train"] - len(glob.glob(f"{FEAT_DIR}/positive_train/my_voice_*.wav")), counts["negative_train"]) < 0.95 * number_of_examples or \
       min(counts["positive_test"], counts["negative_test"]) < 0.95 * n_val:
        raise RuntimeError("Clip generation stopped early — re-run this cell to continue it.")
    open(VOICE_MARK, "w").write(str(add_my_voice()))
    step("2/3 Adding noise and echo, computing features", "--augment_clips", "--overwrite")
    if not features_ready():
        raise RuntimeError("Feature files are incomplete — re-run this cell.")
    save_checkpoint()

onnx_path = f"{OUT_DIR}/{MODEL}.onnx"
if os.path.exists(onnx_path):                       # keep the previous model
    kept = f"{OUT_DIR}/{MODEL}_before_{time.strftime('%m%d_%H%M%S')}.onnx"
    os.rename(onnx_path, kept)
    print("Previous model kept as", os.path.basename(kept))
step(f"3/3 Training (penalty {false_activation_penalty})", "--train_model")
if not os.path.exists(onnx_path):
    raise RuntimeError("Training finished without saving a model — see the output above.")
if IN_COLAB and save_to_drive:
    shutil.copy(onnx_path, f"{DRIVE_DIR}/{MODEL}_penalty{false_activation_penalty}.onnx")
print(f"\nDone: {onnx_path}")
print("Recall above ~0.5 with false positives per hour under ~0.5 is a good result. "
      "Your own test with `python wakeword_tools.py test` is what really counts.")

In [ ]:
# @title 9. Get the model
# @markdown Local PC: optionally the path of the assistant's wakeword folder (for example `/mnt/f/E-book/AIAssistant/wakeword`), to copy the model straight there.
wakeword_folder = ""  # @param {type:"string"}
onnx_path = f"{OUT_DIR}/{MODEL}.onnx"
if wakeword_folder:
    shutil.copy(onnx_path, os.path.join(wakeword_folder, f"{MODEL}.onnx"))
    print("Copied to", os.path.join(wakeword_folder, f"{MODEL}.onnx"))
elif IN_COLAB:
    from google.colab import files
    files.download(onnx_path)
    print(f"Downloading {MODEL}.onnx — put it in the assistant's wakeword folder and restart it.")
else:
    print("Model:", onnx_path, "— copy it to the assistant's wakeword folder.")

## Running locally (Windows + WSL2)

Your PC needs an NVIDIA GPU (8 GB+ VRAM is plenty) and about **70 GB free** on the drive WSL lives on (C: by default).

1. **Install WSL2 with Ubuntu** (skip if you have it). In PowerShell as administrator, run `wsl --install -d Ubuntu-24.04`, then restart and create your Ubuntu username and password.
2. **Check the GPU from Ubuntu:** `nvidia-smi` should list your card. That works through the normal Windows NVIDIA driver; don't install a separate driver inside Ubuntu.
3. **Install the tools and Jupyter** (in the Ubuntu terminal):
   ```bash
   sudo apt update && sudo apt install -y espeak-ng libespeak-ng-dev ffmpeg git curl python3-venv
   python3 -m venv ~/nb && ~/nb/bin/pip install jupyterlab numpy pyyaml
   ~/nb/bin/jupyter lab --notebook-dir ~
   ```
4. Open the link it prints in your Windows browser, upload this notebook (drag it into the file list), and run the cells top to bottom.
5. In cell 1, set `room_noise_zip` and `my_voice_zip` to the zips' paths as WSL sees them: `F:\E-book\AIAssistant\wakeword\my_voice.zip` becomes `/mnt/f/E-book/AIAssistant/wakeword/my_voice.zip`. In cell 9, set `wakeword_folder` to `/mnt/f/E-book/AIAssistant/wakeword` to copy the model straight into the assistant.

Everything lives in `~/wakeword_training`, so a retrain later skips straight to the training step. Delete that folder when you're done (it's about 40 GB).